# 실습과제 멀티에이전트 20260903

In [ ]:
import os
from dotenv import load_dotenv

# 1. 환경 변수 로드 (.env에 OPENAI_API_KEY, TAVILY_API_KEY 필요)
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END, MessagesState

# ==========================================
# 2. 에이전트별 시스템 프롬프트 정의 (과제 가이드 반영)
# ==========================================
RESEARCH_SYS_PROMPT = """당신은 10년 차 시니어 시장 분석가입니다.
사용자가 제시한 [프로젝트 주제]에 대해 다음 세 가지를 반드시 포함하여 리포트를 작성하세요.

1. 현재 시장 트렌드 및 수요 분석
2. 예상 타겟 고객층 (연령, 직업, 특징)
3. 잠재적 경쟁사 또는 유사 서비스의 장단점

사실에 기반한 논조로, 불필요한 서론 없이 핵심만 요약해 주세요."""

DESIGN_SYS_PROMPT = """당신은 트렌드를 선도하는 Product 크리에이티브 디렉터입니다.
사용자가 제시한 [프로젝트 주제]를 바탕으로 서비스의 시각적 컨셉을 기획하세요.
다음 항목을 반드시 포함해야 합니다.

1. 핵심 디자인 키워드 3가지 (예: 미니멀, 레트로, 신뢰감 등)
2. 핵심 유저 인터페이스(UI) 포인트 2가지

개발자와 기획자가 바로 상상할 수 있도록 구체적이고 감각적으로 묘사하세요."""

PM_SYS_PROMPT = """당신은 프로젝트를 총괄하는 헤드 PM(Product Manager)입니다.
아래 제공된 자료조사 리포트와 디자인 방향성을 종합하여 최종 '원페이지 기획서(1-Page PRD)'를 작성하세요.

기획서는 다음 구조를 따르세요:
1. 서비스 한 줄 소개
2. 타겟 고객 및 해결하려는 문제 (자료조사 바탕)
3. 핵심 기능 및 시각적 컨셉 (디자인 방향성 바탕)
4. 다음 액션 플랜 (PM의 관점에서 제안)
"""

# ==========================================
# 3. 도구 및 모델 준비 & 에이전트 3종 생성
# ==========================================
# 검색 도구 (Tavily)
tavily_tool = TavilySearchResults(max_results=3)

# 1) 리서치 에이전트 (Tavily 도구 사용)
research_agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[tavily_tool],
    system_prompt=RESEARCH_SYS_PROMPT,
)

# 2) 디자인 에이전트 (Tavily 도구 사용)
design_agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[tavily_tool],
    system_prompt=DESIGN_SYS_PROMPT,
)

# 3) PM 에이전트 (도구 없음, 과제 가이드에 따라 상위 모델 사용)
pm_llm = init_chat_model("gpt-5.6-luna", temperature=0.2)


# ==========================================
# 4. StateGraph 상태(State) 정의
# ==========================================
class AgentState(MessagesState):
    """
    MessagesState를 상속받아 대화 히스토리(messages)를 유지하고,
    병렬 노드의 출력 결과를 각각 저장할 필드를 추가합니다.
    """

    research_result: str
    design_result: str


# ==========================================
# 5. Graph 노드(Node) 함수 정의
# ==========================================
def research_node(state: AgentState):
    """에이전트 1: 자료조사 (병렬 실행)"""
    user_input = state["messages"][-1].content
    response = research_agent.invoke({"messages": [HumanMessage(user_input)]})
    return {"research_result": response["messages"][-1].content}


def design_node(state: AgentState):
    """에이전트 2: 시각 디자인 기획 (병렬 실행)"""
    user_input = state["messages"][-1].content
    response = design_agent.invoke({"messages": [HumanMessage(user_input)]})
    return {"design_result": response["messages"][-1].content}


def pm_node(state: AgentState):
    """에이전트 3: PM 종합 및 최종 PRD 작성 (Fan-in)"""
    user_input = state["messages"][0].content

    pm_prompt = f"""요청 주문: {user_input}
---
[자료조사 리포트]
{state['research_result']}
---
[디자인 방향성]
{state['design_result']}
"""

    response = pm_llm.invoke(
        [SystemMessage(PM_SYS_PROMPT), HumanMessage(pm_prompt)]
    )

    # 최종 PRD를 메시지 상태에 추가하여 반환
    return {"messages": [response]}


# ==========================================
# 6. Graph 빌드 및 컴파일 (START -> 병렬 -> PM -> END)
# ==========================================
builder = StateGraph(AgentState)

# 노드 등록
builder.add_node("research_node", research_node)
builder.add_node("design_node", design_node)
builder.add_node("pm_node", pm_node)

# 엣지 연결 (수업에서 다룬 Fan-out / Fan-in 패턴)
# 1) START에서 두 노드로 동시 연결 -> 병렬 실행
builder.add_edge(START, "research_node")
builder.add_edge(START, "design_node")

# 2) 두 노드의 결과를 PM 노드로 모음 (Fan-in)
builder.add_edge("research_node", "pm_node")
builder.add_edge("design_node", "pm_node")

# 3) PM 노드 완료 후 종료
builder.add_edge("pm_node", END)

# 컴파일
app = builder.compile()


# ==========================================
# 7. 실행 테스트
# ==========================================
if __name__ == "__main__":
    user_prompt = "새로운 스마트 안경을 만들고 싶어."

    print(f"🚀 요청 시작: '{user_prompt}'\n")

    result = app.invoke({"messages": [HumanMessage(user_prompt)]})

    print("=" * 60)
    print("📋 [최종 PM 1-Page PRD 리포트]")
    print("=" * 60)
    print(result["messages"][-1].content)

🚀 요청 시작: '새로운 스마트 안경을 만들고 싶어.'

📋 [최종 PM 1-Page PRD 리포트]
1. 서비스 한 줄 소개
- “투명하고 직관적인 AI·AR 보조를 제공하는 하이브리드 스마트 안경 — 업무 생산성과 일상 편의성을 동시에 높이는 경량·프라이버시 중심의 웨어러블.”

2. 타겟 고객 및 해결하려는 문제 (자료조사 바탕)
- 타겟 고객
  - 연령: 20대 후반 ~ 40대 초중반, 기술 수용도가 높은 전문가·크리에이터·원격근무자
  - 직업군: IT/테크, 원격 협업 사용자, 교육자·강사, 의료진, 산업 현장 엔지니어(필드 서비스)
  - 특징: 실시간 정보·원격지시·멀티태스킹으로 생산성 향상을 원하는 사용자
- 해결하려는 문제
  - 정보 접근성: 손을 쓰기 힘든 상황에서 즉시 참조할 수 있는 컨텍스트 기반 정보 부재
  - 생산성 손실: 원격 협업·지시 전달 시 시선·손동작으로 인한 비효율
  - 사용성·착용성: 무겁고 눈을 가리는 기존 AR 기기의 불편함
  - 배터리·프라이버시: 짧은 사용시간과 데이터/시선 트래킹에 대한 신뢰 부족
  - 디자인·수용성: 일상 착용을 망설이게 하는 패션·심미성 부족

3. 핵심 기능 및 시각적 컨셉 (디자인 방향성 바탕)
- 핵심 기능 (우선순위: MVP → 단계적 확장)
  - MVP(반드시 포함)
    - 투명 AR HUD: 시야 한켠에 반투명 정보 오버레이(알림, 미니맵, 텍스트 요약)
    - 음성·AI 어시스턴트: 자연어 질의/요약/명령 (로컬/클라우드 하이브리드)
    - 간결 제스처 컨트롤: 다리/렌즈 주변 스와이프·탭으로 탐색·확인
    - 안전·프라이버시 모드: 카메라·마이크 물리적 차단 + 사용자 데이터 암호화
    - 기본 통신: 블루투스, Wi‑Fi, 스마트폰 연동 알림·콜 처리
  - 1차 업그레이드
    - 원격 협업 AR: 원격지 화면 공유·핸드·포인터 오버레이
    - 현장용 데이터 오버레이: 실시간 센서·매뉴얼·체크리스트 표시
    - 배터리 최적화 모드: 보조